In [151]:
import sys
import os
import mysql.connector
import pandas as pd
import numpy as np
sys.path.append(os.path.abspath('..'))
from config import get_db_config
import warnings
warnings.filterwarnings('ignore')

In [152]:
# Connect ke database config
config = get_db_config()
# Ambil host dari salah satu config (misal db_old)
print(f'Database config loaded: {config["db_old"]["host"]}')

# Connect ke DB Lama
db_old = mysql.connector.connect(**config['db_old'])
cursor_old = db_old.cursor(dictionary=True)
print(f'Connected to old database: {config["db_old"]["database"]}')

# Connect ke DB Baru
db_new = mysql.connector.connect(**config['db_new'])
cursor_new = db_new.cursor(dictionary=True)
print(f'Connected to new database: {config["db_new"]["database"]}')

db_future = mysql.connector.connect(**config['db_future'])
cursor_future = db_future.cursor(dictionary=True)
print(f'Connected to future database: {config["db_future"]["database"]}')


Database config loaded: localhost
Connected to old database: dataleap_juni
Connected to new database: dataleap_v5_migration
Connected to future database: db_future


In [153]:
import pandas as pd
import pickle

# =========================================================
# 1. EXTRACT DATA DARI DB LAMA
# =========================================================
print("📥 Mengambil data jadwal dari database lama...")
cursor_old.execute("SELECT * FROM jadwal")
data_jadwal_old = cursor_old.fetchall()
df_raw_jadwal = pd.DataFrame(data_jadwal_old)
print(f"Total data asli di DB lama: {len(df_raw_jadwal)} baris")

# Extract tabel jadwal
df_jadwal_lama = pd.read_sql("SELECT * FROM jadwal", db_old)
import pandas as pd
import pickle

# =========================================================
# 2. TRANSFORMASI TABEL: jadwal (seperti kode Anda)
# =========================================================
print("⚡ Melakukan transformasi tabel 'jadwal'...")

# A. Pembersihan metode belajar
def clean_mode_belajar(val):
    if pd.isna(val) or not str(val).strip():
        return 'Offline'
    s = str(val).strip().capitalize()
    if s in ('Online', 'Offline', 'Hybrid'):
        return s
    return 'Offline'

# B. Pembersihan status arsip
def clean_status_arsip(val):
    try:
        if pd.isna(val):
            return 0
        return int(float(val))
    except:
        return 0

# C. Saring (filter out) jadwal percobaan
df_jadwal_clean = df_raw_jadwal[~df_raw_jadwal['idperiode'].isin(['P00094', 'P00104'])].copy()
skipped_count = len(df_raw_jadwal) - len(df_jadwal_clean)
print(f"ℹ️ Menyaring {skipped_count} data jadwal percobaan. Sisa data: {len(df_jadwal_clean)} baris")

# Siapkan dataframe untuk insert (tanpa id)
df_jadwal_insert = pd.DataFrame({
    'id_kursus': df_jadwal_clean['idpendkursus'],
    'id_periode': df_jadwal_clean['idperiode'],
    'id_level': df_jadwal_clean['idlevel'],
    'id_sesi': df_jadwal_clean['idsesi'],
    'metode_belajar_jadwal': df_jadwal_clean['mode_belajar'].apply(clean_mode_belajar),
    'nama_rombel': df_jadwal_clean['groupwa'].fillna('').str.strip(),
    'status_arsip': df_jadwal_clean['status_archive'].apply(clean_status_arsip),
    'tempat': df_jadwal_clean['tempat'].fillna('Ruang Kelas').replace('', 'Ruang Kelas').str.strip()
})

# Simpan urutan old_id (sesuai urutan df_jadwal_insert)
old_id_list = df_jadwal_clean['idjadwal'].tolist()

# =========================================================
# BUAT ID BARU (SIMULASI AUTO INCREMENT) & MAPPING
# =========================================================
print("🔢 Membuat ID baru (auto increment simulasi)...")
# Tambahkan kolom 'id' dengan angka urut mulai dari 1
df_jadwal_insert.insert(0, 'id', range(1, len(df_jadwal_insert) + 1))

# Buat mapping old -> new
mapping_id_jadwal = dict(zip(old_id_list, df_jadwal_insert['id'].tolist()))
print(f"✅ Mapping ID jadwal selesai. Jumlah: {len(mapping_id_jadwal)}")

# Simpan mapping ke file pickle (opsional)
with open('mapping_id_jadwal.pkl', 'wb') as f:
    pickle.dump(mapping_id_jadwal, f)
print("💾 Mapping disimpan ke 'mapping_id_jadwal.pkl'")

# =========================================================
# 3. TRANSFORMASI TABEL BARU: jadwal_hari (menggunakan mapping)
# =========================================================
print("⚡ Memisahkan kolom 'hari' ke tabel 'jadwal_hari'...")

hari_rows = []
for idx, row in df_jadwal_clean.iterrows():
    old_id = row['idjadwal']
    hari_string = row['hari']
    if pd.isna(hari_string) or not str(hari_string).strip():
        continue
    for hari in [h.strip() for h in hari_string.split(',') if h.strip()]:
        hari_rows.append({
            'id_jadwal': old_id,   # masih old id
            'nama_hari': hari
        })

df_jadwal_hari = pd.DataFrame(hari_rows)

# Ganti id_jadwal dengan ID baru menggunakan mapping
df_jadwal_hari['id_jadwal'] = df_jadwal_hari['id_jadwal'].map(mapping_id_jadwal)
# Hapus baris yang tidak punya mapping (jika ada)
df_jadwal_hari = df_jadwal_hari.dropna(subset=['id_jadwal'])
df_jadwal_hari['id_jadwal'] = df_jadwal_hari['id_jadwal'].astype(int)

print(f"✓ Tabel 'jadwal_hari' siap. Shape: {df_jadwal_hari.shape}")

# =========================================================
# 4. SIMPAN SEMUA DATA UNTUK TAHAP SELANJUTNYA
# =========================================================
fase_4_afrida = {
    'jadwal': df_jadwal_insert,          # sudah punya id baru
    'jadwal_old_ids': old_id_list,       # urutan lama (untuk referensi)
    'jadwal_hari': df_jadwal_hari,       # sudah pakai id baru
    'mapping_id_jadwal': mapping_id_jadwal,
    # nanti tambahkan 'jadwal_detail', 'jadwal_pengajar', 'jadwal_siswa', dll.
}


📥 Mengambil data jadwal dari database lama...
Total data asli di DB lama: 558 baris
⚡ Melakukan transformasi tabel 'jadwal'...
ℹ️ Menyaring 2 data jadwal percobaan. Sisa data: 556 baris
🔢 Membuat ID baru (auto increment simulasi)...
✅ Mapping ID jadwal selesai. Jumlah: 556
💾 Mapping disimpan ke 'mapping_id_jadwal.pkl'
⚡ Memisahkan kolom 'hari' ke tabel 'jadwal_hari'...
✓ Tabel 'jadwal_hari' siap. Shape: (982, 2)


In [154]:
display(df_jadwal_insert.head())

,id,id_kursus,id_periode,id_level,id_sesi,metode_belajar_jadwal,nama_rombel,status_arsip,tempat
0,1,K00001,P00006,L00017,S00002,Online,01 GOGO 3B SR2 (ERICA),1,Ruang Kelas 4
1,2,K00001,P00006,L00024,S00002,Offline,02 SO 1C SR2 (QORIN),1,Ruang Kelas 5
2,3,K00001,P00006,L00023,S00001,Offline,03 SO 1B SR1 (TATIK),1,Ruang Kelas 1
3,4,K00001,P00006,L00025,S00003,Offline,04 SO 2A SR3 (TATIK),1,Ruang Kelas 1
4,5,K00001,P00006,L00014,S00003,Offline,05 GOGO 1B SelK3 (ERICA),1,Ruang Kelas 4


In [155]:
# =========================================================
# 4. TRANSFORMASI TABEL: jadwal_detail (sumber: jadwal_detil)
# =========================================================
print("⚡ Melakukan transformasi tabel 'jadwal_detail'...")

# Ambil data mentah
df_detil_lama = pd.read_sql("SELECT * FROM jadwal_detil", db_old)
print(f"  Data mentah: {len(df_detil_lama)} baris")

# Filter hanya idjadwal yang valid (yang lolos filter di jadwal)
valid_jadwal_ids = set(df_jadwal_clean['idjadwal'])
df_detil_lama = df_detil_lama[df_detil_lama['idjadwal'].isin(valid_jadwal_ids)].copy()
print(f"  Setelah filter idjadwal: {len(df_detil_lama)} baris")

# Simpan old_id detail (urutan akan sama dengan df setelah filter)
old_detail_ids = df_detil_lama['idjadwaldetil'].tolist()

# Buat dataframe untuk insert (tanpa id, akan auto increment)
df_jadwal_detail_insert = pd.DataFrame({
    'judul': df_detil_lama['title'].fillna('').astype(str),
    'deskripsi': df_detil_lama['description'].fillna('').astype(str),
    'url_jadwal_detail': df_detil_lama['url'].fillna('').astype(str),
    'id_jadwal': df_detil_lama['idjadwal'],   # masih old_id
    'label_warna': df_detil_lama['color'].fillna('').astype(str),
    'penanda_mulai': pd.to_datetime(df_detil_lama['start'], errors='coerce').dt.date,
    'penanda_selesai': pd.to_datetime(df_detil_lama['end'], errors='coerce').dt.date,
})

# Kolom tambahan default
df_jadwal_detail_insert['id_mitra'] = None
df_jadwal_detail_insert['id_sesi_override'] = None
df_jadwal_detail_insert['status_detail'] = 'Scheduled'
df_jadwal_detail_insert['source_type'] = 'Generated'
df_jadwal_detail_insert['original_jadwal_detail_id'] = None
df_jadwal_detail_insert['has_operational_data'] = 0
df_jadwal_detail_insert['last_generated_at'] = None
df_jadwal_detail_insert['created_at'] = pd.Timestamp.now()
df_jadwal_detail_insert['updated_at'] = pd.Timestamp.now()

# Cleaning deskripsi & URL
df_jadwal_detail_insert['deskripsi'] = (
    df_jadwal_detail_insert['deskripsi']
    .fillna('')
    .replace('', 'Tidak ada deskripsi')
    .str.strip()
)
df_jadwal_detail_insert.loc[df_jadwal_detail_insert['deskripsi'] == '', 'deskripsi'] = 'Tidak ada deskripsi'

df_jadwal_detail_insert['url_jadwal_detail'] = (
    df_jadwal_detail_insert['url_jadwal_detail']
    .fillna('')
    .replace('-', 'Link belum tersedia')
    .str.strip()
)
df_jadwal_detail_insert.loc[df_jadwal_detail_insert['url_jadwal_detail'] == '', 'url_jadwal_detail'] = 'Link belum tersedia'

print(f"✓ df_jadwal_detail_insert siap. Shape: {df_jadwal_detail_insert.shape}")

# =========================================================
# GANTI id_jadwal dengan ID baru dari mapping
# =========================================================
df_jadwal_detail_insert['id_jadwal'] = df_jadwal_detail_insert['id_jadwal'].map(mapping_id_jadwal)
# Hapus baris yang tidak terpetakan (seharusnya tidak ada)
df_jadwal_detail_insert = df_jadwal_detail_insert.dropna(subset=['id_jadwal'])
df_jadwal_detail_insert['id_jadwal'] = df_jadwal_detail_insert['id_jadwal'].astype(int)

# =========================================================
# BUAT ID BARU UNTUK JADWAL_DETAIL (simulasi auto increment)
# =========================================================
df_jadwal_detail_insert.insert(0, 'id', range(1, len(df_jadwal_detail_insert) + 1))

# Mapping old_detail_id -> new_detail_id
mapping_id_jadwal_detail = dict(zip(old_detail_ids, df_jadwal_detail_insert['id'].tolist()))
print(f"✅ Mapping ID jadwal_detail selesai. Jumlah: {len(mapping_id_jadwal_detail)}")

# =========================================================
# SIMPAN KE DICTIONARY & FILE PICKLE
# =========================================================
# 14. Simpan ke fase_4_afrida
fase_4_afrida['jadwal_detail'] = df_jadwal_detail_insert
fase_4_afrida['jadwal_detail_old_ids'] = list(mapping_id_jadwal_detail.keys())
fase_4_afrida['mapping_id_jadwal_detail'] = mapping_id_jadwal_detail

⚡ Melakukan transformasi tabel 'jadwal_detail'...
  Data mentah: 17322 baris
  Setelah filter idjadwal: 17312 baris
✓ df_jadwal_detail_insert siap. Shape: (17312, 16)
✅ Mapping ID jadwal_detail selesai. Jumlah: 17312


In [156]:
# =========================================================
# VERIFIKASI FOREIGN KEY: jadwal_detail -> jadwal
# =========================================================
print("="*70)
print("🔍 MEMERIKSA RELASI FK: df_jadwal_detail_insert -> df_jadwal_insert")
print("="*70)

# Ambil set ID jadwal baru (primary key)
pk_jadwal = set(df_jadwal_insert['id'])

# Set ID jadwal di tabel detail (foreign key)
fk_jadwal_detail = set(df_jadwal_detail_insert['id_jadwal'])

# 1. Cek NULL pada kolom id_jadwal
null_count = df_jadwal_detail_insert['id_jadwal'].isna().sum()
print(f"1. Jumlah nilai NULL pada kolom 'id_jadwal' di detail: {null_count}")
if null_count > 0:
    print("   ⚠️ Ada NULL! Perbaiki mapping.")
else:
    print("   ✅ Tidak ada NULL.")

# 2. Cek apakah semua FK ada di PK
invalid_fk = fk_jadwal_detail - pk_jadwal
print(f"2. Jumlah id_jadwal di detail yang TIDAK ADA di tabel jadwal: {len(invalid_fk)}")
if len(invalid_fk) > 0:
    print(f"   ❌ ID tidak valid (contoh 5): {list(invalid_fk)[:5]}")
else:
    print("   ✅ Semua FK valid.")

# 3. Statistik jumlah data
total_detail = len(df_jadwal_detail_insert)
total_jadwal = len(pk_jadwal)
print(f"3. Total data jadwal_detail: {total_detail}")
print(f"   Total data jadwal: {total_jadwal}")
print(f"   Jumlah unik id_jadwal di detail: {len(fk_jadwal_detail)}")

# 4. Cek orphan (detail tanpa parent) – seharusnya 0
orphan = df_jadwal_detail_insert[~df_jadwal_detail_insert['id_jadwal'].isin(pk_jadwal)]
print(f"4. Jumlah data detail orphan (tanpa parent): {len(orphan)}")
if len(orphan) > 0:
    print("   ❌ Ada orphan! Periksa data berikut:")
    display(orphan[['id', 'id_jadwal', 'judul']].head(10))

# 5. Tampilkan contoh join (beberapa baris) untuk inspeksi visual
print("\n5. Contoh data detail dengan parent-nya (5 baris pertama):")
# Gabungkan dengan jadwal untuk melihat nama rombel atau info lain sebagai sampel
sample_join = df_jadwal_detail_insert[['id', 'id_jadwal', 'judul']].head(5).merge(
    df_jadwal_insert[['id', 'nama_rombel', 'metode_belajar_jadwal']],
    left_on='id_jadwal', right_on='id', how='left'
)
display(sample_join)

print("="*70)
if null_count == 0 and len(invalid_fk) == 0 and len(orphan) == 0:
    print("✅ VERIFIKASI LULUS: Semua FK terhubung dengan benar.")
else:
    print("❌ VERIFIKASI GAGAL: Ada masalah pada relasi FK. Perbaiki sebelum lanjut.")

🔍 MEMERIKSA RELASI FK: df_jadwal_detail_insert -> df_jadwal_insert
1. Jumlah nilai NULL pada kolom 'id_jadwal' di detail: 0
   ✅ Tidak ada NULL.
2. Jumlah id_jadwal di detail yang TIDAK ADA di tabel jadwal: 0
   ✅ Semua FK valid.
3. Total data jadwal_detail: 17312
   Total data jadwal: 556
   Jumlah unik id_jadwal di detail: 556
4. Jumlah data detail orphan (tanpa parent): 0

5. Contoh data detail dengan parent-nya (5 baris pertama):


,id_x,id_jadwal,judul,id_y,nama_rombel,metode_belajar_jadwal
0,1,8,08 SO 2A SelK3 (GETA),8,08 SO 2A SelK3 (GETA),Online
1,2,8,08 SO 2A SelK3 (GETA),8,08 SO 2A SelK3 (GETA),Online
2,3,8,08 SO 2A SelK3 (GETA),8,08 SO 2A SelK3 (GETA),Online
3,4,8,08 SO 2A SelK3 (GETA),8,08 SO 2A SelK3 (GETA),Online
4,5,8,08 SO 2A SelK3 (GETA),8,08 SO 2A SelK3 (GETA),Online


✅ VERIFIKASI LULUS: Semua FK terhubung dengan benar.


In [157]:
# =========================================================
# 5. TRANSFORMASI TABEL: jadwal_pengajar (sumber: jadwal_pengajar)
# =========================================================
print("="*70)
print("⚡ Memproses tabel 'jadwal_pengajar'")
print("="*70)

# 1. Ambil data mentah
df_pengajar_lama = pd.read_sql("SELECT * FROM jadwal_pengajar", db_old)
print(f"Data mentah: {len(df_pengajar_lama)} baris")

# 2. Simpan old_id untuk tracing
df_pengajar_lama['old_id_pengajar'] = df_pengajar_lama['idpengajar'].astype(str)
df_pengajar_lama['old_id_jadwal']   = df_pengajar_lama['idjadwal'].astype(str)

# 3. Filter: hanya yang idjadwal-nya ada di mapping (parent valid)
df_pengajar_lama = df_pengajar_lama[df_pengajar_lama['old_id_jadwal'].isin(mapping_id_jadwal.keys())].copy()
print(f"Setelah filter parent valid: {len(df_pengajar_lama)} baris")

# 4. Ambil id_user (prioritas idusers, fallback ke idguru jika ada)
if 'idusers' in df_pengajar_lama.columns:
    id_user_col = 'idusers'
elif 'idguru' in df_pengajar_lama.columns:
    id_user_col = 'idguru'
else:
    id_user_col = None
    print("⚠️ Kolom id_user tidak ditemukan. Semua id_user akan diisi None.")

# Buat dataframe sementara
df_temp = pd.DataFrame({
    'old_id_pengajar': df_pengajar_lama['old_id_pengajar'],
    'old_id_jadwal':   df_pengajar_lama['old_id_jadwal'],
    'id_user':         df_pengajar_lama[id_user_col].astype(str) if id_user_col else np.nan
})

# Bersihkan id_user: ubah 'nan', 'None', '' menjadi NaN
df_temp['id_user'] = df_temp['id_user'].replace(['nan', 'None', ''], np.nan)
df_temp['id_user'] = df_temp['id_user'].fillna(np.nan)

# Hapus baris yang tidak punya id_user (karena FK ke users)
before_drop_user = len(df_temp)
df_temp = df_temp.dropna(subset=['id_user'])
print(f"Baris tanpa id_user dibuang: {before_drop_user - len(df_temp)}")

# 5. Terapkan mapping id_jadwal
df_temp['id_jadwal'] = df_temp['old_id_jadwal'].map(mapping_id_jadwal)
before_drop_map = len(df_temp)
df_temp = df_temp.dropna(subset=['id_jadwal'])
print(f"Orphan (id_jadwal tidak valid) dibuang: {before_drop_map - len(df_temp)}")

# 6. Inner join dengan df_jadwal_insert untuk memastikan FK valid
df_jadwal_pk = df_jadwal_insert[['id']].copy()
df_temp = df_temp.merge(df_jadwal_pk, left_on='id_jadwal', right_on='id', how='inner')
print(f"Setelah inner join dengan jadwal: {len(df_temp)} baris")
df_temp = df_temp.drop(columns=['id'])  # hapus kolom 'id' hasil merge (karena kita pakai id_jadwal)

# 7. Tambahkan kolom created_at, updated_at
df_temp['created_at'] = pd.Timestamp.now()
df_temp['updated_at'] = pd.Timestamp.now()

# 8. Pilih kolom yang diperlukan (tanpa kolom old)
df_jadwal_pengajar = df_temp[['id_jadwal', 'id_user', 'created_at', 'updated_at']].copy()

# 9. Tambahkan id (auto increment) untuk tabel jadwal_pengajar
df_jadwal_pengajar.insert(0, 'id', range(1, len(df_jadwal_pengajar) + 1))

# (Opsional) Rename kolom 'id' menjadi 'id_jadwal_pengajar' jika skema baru mengharapkan nama itu
# df_jadwal_pengajar = df_jadwal_pengajar.rename(columns={'id': 'id_jadwal_pengajar'})

print(f"✅ jadwal_pengajar siap. Shape: {df_jadwal_pengajar.shape}")

# 10. Simpan ke fase_4_afrida
fase_4_afrida['jadwal_pengajar'] = df_jadwal_pengajar

⚡ Memproses tabel 'jadwal_pengajar'
Data mentah: 651 baris
Setelah filter parent valid: 650 baris
Baris tanpa id_user dibuang: 0
Orphan (id_jadwal tidak valid) dibuang: 0
Setelah inner join dengan jadwal: 650 baris
✅ jadwal_pengajar siap. Shape: (650, 5)


In [158]:
pk_jadwal = set(df_jadwal_insert['id'])
fk_pengajar = set(df_jadwal_pengajar['id_jadwal'])
invalid = fk_pengajar - pk_jadwal

print(f"Total pengajar: {len(df_jadwal_pengajar)}")
print(f"Total jadwal: {len(df_jadwal_insert)}")
print(f"Invalid FK ke jadwal: {len(invalid)}")
if len(invalid) == 0:
    print("✅ Semua FK ke jadwal valid!")
else:
    print("❌ Ada invalid, periksa mapping.")

Total pengajar: 650
Total jadwal: 556
Invalid FK ke jadwal: 0
✅ Semua FK ke jadwal valid!


In [159]:
import pickle

# Load mapping siswa
with open('mapping_siswa.pkl', 'rb') as f:
    mapping_siswa = pickle.load(f)

print("Type mapping_siswa:", type(mapping_siswa))
print("Panjang mapping_siswa:", len(mapping_siswa))

# Lihat struktur
if isinstance(mapping_siswa, dict):
    # Cek tipe data key dan value
    keys = list(mapping_siswa.keys())[:5]
    values = [mapping_siswa[k] for k in keys]
    print("Contoh key:", keys)
    print("Contoh value:", values)
    print("Tipe key:", type(keys[0]) if keys else "None")
    print("Tipe value:", type(values[0]) if values else "None")
    
    # Cek apakah value berupa dict (mungkin nested)
    if isinstance(values[0], dict):
        print("⚠️ Mapping memiliki nested structure!")
        print("  Contoh nested:", values[0])
        
        # Ambil mapping yang sebenarnya (misal dari key 'id_siswa' atau lainnya)
        # Sesuaikan dengan struktur yang terlihat
else:
    print("Bukan dictionary! Ini adalah:", type(mapping_siswa))

Type mapping_siswa: <class 'pandas.core.frame.DataFrame'>
Panjang mapping_siswa: 1500
Bukan dictionary! Ini adalah: <class 'pandas.core.frame.DataFrame'>


In [160]:
import pandas as pd
import pickle

# =========================================================
# 2. LOAD MAPPING
# =========================================================
print("\n" + "="*70)
print("📂 Memuat mapping untuk jadwal_siswa")
print("="*70)

# Load mapping jadwal dari fase_4_afrida.pkl
with open('fase_4_afrida.pkl', 'rb') as f:
    data_fase4 = pickle.load(f)
mapping_id_jadwal = data_fase4['mapping_id_jadwal']
print(f"✅ Mapping id_jadwal: {len(mapping_id_jadwal)} entri")

# Load mapping siswa dari mapping_siswa.pkl
with open('mapping_siswa.pkl', 'rb') as f:
    mapping_siswa_df = pickle.load(f)

if isinstance(mapping_siswa_df, pd.DataFrame):
    kolom_old = mapping_siswa_df.columns[0]
    kolom_new = mapping_siswa_df.columns[1]
    mapping_id_siswa = dict(zip(
        mapping_siswa_df[kolom_old].astype(str),
        mapping_siswa_df[kolom_new]
    ))
    print(f"✅ Mapping siswa: {len(mapping_id_siswa)} entri")
else:
    mapping_id_siswa = mapping_siswa_df
    print(f"✅ Mapping siswa: {len(mapping_id_siswa)} entri")

# =========================================================
# 3. PROSES DATA jadwal_siswa
# =========================================================
print("\n" + "="*70)
print("⚡ Proses jadwal_siswa (semua tanggal nullable)")
print("="*70)

# Ambil data mentah dari DB lama
cursor_old.execute("SELECT * FROM jadwal_siswa")
rows = cursor_old.fetchall()
columns = [desc[0] for desc in cursor_old.description]
df_siswa_lama = pd.DataFrame(rows, columns=columns)
print(f"Data mentah: {len(df_siswa_lama)} baris")

# Buat dataframe sementara
df_temp = pd.DataFrame({
    'old_id_siswa': df_siswa_lama['idjadwal_siswa'].astype(str),
    'old_id_jadwal': df_siswa_lama['idjadwal'].astype(str),
    'id_siswa_old': df_siswa_lama['idsiswa'].astype(str),
    'tambahan_sesi': pd.to_numeric(df_siswa_lama['tambahan_sesi'], errors='coerce').fillna(0).astype(int),
    'tambahan_keterangan': df_siswa_lama['tambahan_ket'].fillna('Belum ada keterangan').astype(str),
    'status_keluar': pd.to_numeric(df_siswa_lama['is_keluar'], errors='coerce').fillna(0).astype(int)
})

# =========================================================
# 4. KONVERSI TANGGAL (biarkan NULL)
# =========================================================
print("\n📌 Konversi tanggal (NULL dibiarkan NULL)...")

# Konversi ke date (tanpa waktu) - NULL tetap NULL
df_temp['tanggal_mulai'] = pd.to_datetime(df_siswa_lama['tgl_mulai'], errors='coerce').dt.date
df_temp['tanggal_keluar'] = pd.to_datetime(df_siswa_lama['tgl_keluar'], errors='coerce').dt.date
df_temp['tanggal_aktif'] = pd.to_datetime(df_siswa_lama['tgl_aktif'], errors='coerce').dt.date

# Statistik NULL
print(f"  tanggal_mulai NULL: {df_temp['tanggal_mulai'].isna().sum()}")
print(f"  tanggal_keluar NULL: {df_temp['tanggal_keluar'].isna().sum()}")
print(f"  tanggal_aktif NULL: {df_temp['tanggal_aktif'].isna().sum()}")

# =========================================================
# 5. MAPPING id_jadwal
# =========================================================
print("\n📌 Mapping id_jadwal...")
df_temp['id_jadwal'] = df_temp['old_id_jadwal'].map(mapping_id_jadwal)
before = len(df_temp)
df_temp.dropna(subset=['id_jadwal'], inplace=True)
print(f"  Baris orphan jadwal dibuang: {before - len(df_temp)}")
df_temp['id_jadwal'] = df_temp['id_jadwal'].astype('int64')

# =========================================================
# 6. MAPPING id_siswa
# =========================================================
print("\n📌 Mapping id_siswa...")
df_temp['id_siswa'] = df_temp['id_siswa_old'].map(mapping_id_siswa)
before_siswa = len(df_temp)
df_temp.dropna(subset=['id_siswa'], inplace=True)
print(f"  Baris orphan siswa dibuang: {before_siswa - len(df_temp)}")
df_temp['id_siswa'] = df_temp['id_siswa'].astype('int64')

# =========================================================
# 7. TAMBAHKAN KOLOM DEFAULT
# =========================================================
df_temp['is_acc_rapor'] = 0
df_temp['status_ketuntasan'] = None
df_temp['catatan_ketuntasan_guru'] = None
df_temp['catatan_ketuntasan_admin'] = None
df_temp['ketuntasan_diperbarui_oleh'] = None
df_temp['ketuntasan_diperbarui_pada'] = None

# =========================================================
# 8. FINALISASI
# =========================================================
print("\n📌 Finalisasi...")

# Pilih kolom final
df_jadwal_siswa = df_temp[[
    'id_jadwal',
    'id_siswa',
    'tanggal_mulai',
    'tanggal_keluar',
    'tanggal_aktif',
    'tambahan_sesi',
    'tambahan_keterangan',
    'status_keluar',
    'is_acc_rapor',
    'status_ketuntasan',
    'catatan_ketuntasan_guru',
    'catatan_ketuntasan_admin',
    'ketuntasan_diperbarui_oleh',
    'ketuntasan_diperbarui_pada'
]].copy()

# Tambahkan id internal
df_jadwal_siswa.insert(0, 'id', range(1, len(df_jadwal_siswa) + 1))

# =========================================================
# 9. CEK TIPE DATA & NULL
# =========================================================
print("\n📌 Cek tipe data dan NULL...")
print(f"  tanggal_mulai dtype: {df_jadwal_siswa['tanggal_mulai'].dtype}")
print(f"  tanggal_keluar dtype: {df_jadwal_siswa['tanggal_keluar'].dtype}")
print(f"  tanggal_aktif dtype: {df_jadwal_siswa['tanggal_aktif'].dtype}")

# Cek NULL
null_counts = df_jadwal_siswa.isnull().sum()
if null_counts.sum() > 0:
    print("\n📊 NULL per kolom:")
    print(null_counts[null_counts > 0])
else:
    print("✅ Tidak ada NULL di semua kolom!")

# =========================================================
# 10. HASIL AKHIR
# =========================================================
print(f"\n✅ df_jadwal_siswa siap. Shape: {df_jadwal_siswa.shape}")
print(f"   Kolom: {df_jadwal_siswa.columns.tolist()}")

print("\n📋 Sample data (5 baris):")
print(df_jadwal_siswa[['id', 'id_jadwal', 'id_siswa', 'tanggal_mulai', 'tanggal_keluar', 'tanggal_aktif']].head())

# =========================================================
# 11. SIMPAN KE fase_4_afrida.pkl
# =========================================================
# 6. Simpan ke fase_4_afrida
fase_4_afrida['catatan_kelas_tag'] = df_jadwal_siswa
fase_4_afrida['mapping_id_jadwal_siswa'] = dict(zip(df_temp['old_id_siswa'], df_jadwal_siswa['id']))

print("\n✅ Proses jadwal_siswa selesai!")


📂 Memuat mapping untuk jadwal_siswa
✅ Mapping id_jadwal: 556 entri
✅ Mapping siswa: 1500 entri

⚡ Proses jadwal_siswa (semua tanggal nullable)
Data mentah: 3960 baris

📌 Konversi tanggal (NULL dibiarkan NULL)...
  tanggal_mulai NULL: 2856
  tanggal_keluar NULL: 3924
  tanggal_aktif NULL: 3960

📌 Mapping id_jadwal...
  Baris orphan jadwal dibuang: 2

📌 Mapping id_siswa...
  Baris orphan siswa dibuang: 0

📌 Finalisasi...

📌 Cek tipe data dan NULL...
  tanggal_mulai dtype: object
  tanggal_keluar dtype: object
  tanggal_aktif dtype: datetime64[ns]

📊 NULL per kolom:
tanggal_mulai                 2855
tanggal_keluar                3922
tanggal_aktif                 3958
status_ketuntasan             3958
catatan_ketuntasan_guru       3958
catatan_ketuntasan_admin      3958
ketuntasan_diperbarui_oleh    3958
ketuntasan_diperbarui_pada    3958
dtype: int64

✅ df_jadwal_siswa siap. Shape: (3958, 15)
   Kolom: ['id', 'id_jadwal', 'id_siswa', 'tanggal_mulai', 'tanggal_keluar', 'tanggal_aktif',

In [161]:
# =========================================================
# VERIFIKASI: Cek NULL di df_jadwal_siswa
# =========================================================
print("\n" + "="*70)
print("🔍 CEK NULL DI DATAFRAME jadwal_siswa")
print("="*70)

if 'df_jadwal_siswa' in locals():
    print(f"\n📊 Total data: {len(df_jadwal_siswa)} baris")
    
    # Cek NULL per kolom tanggal
    print("\n📌 NULL per kolom tanggal:")
    print(f"  tanggal_mulai NULL : {df_jadwal_siswa['tanggal_mulai'].isna().sum()}")
    print(f"  tanggal_keluar NULL: {df_jadwal_siswa['tanggal_keluar'].isna().sum()}")
    print(f"  tanggal_aktif NULL : {df_jadwal_siswa['tanggal_aktif'].isna().sum()}")
    
    # Tampilkan sample data dengan NULL
    print("\n📋 Sample data (5 baris dengan NULL):")
    print(df_jadwal_siswa[['id', 'id_jadwal', 'id_siswa', 'tanggal_mulai', 'tanggal_keluar', 'tanggal_aktif']].head())
    
else:
    print("⚠️ df_jadwal_siswa belum dibuat.")


🔍 CEK NULL DI DATAFRAME jadwal_siswa

📊 Total data: 3958 baris

📌 NULL per kolom tanggal:
  tanggal_mulai NULL : 2855
  tanggal_keluar NULL: 3922
  tanggal_aktif NULL : 3958

📋 Sample data (5 baris dengan NULL):
   id  id_jadwal  id_siswa tanggal_mulai tanggal_keluar tanggal_aktif
0   1          3       347           NaT            NaT           NaT
1   2          3       348           NaT            NaT           NaT
2   3          7        79           NaT            NaT           NaT
3   4          7        82           NaT            NaT           NaT
4   5          7       108           NaT            NaT           NaT


In [162]:
# =========================================================
# 7. TRANSFORMASI TABEL: catatan_kelas (sumber: catatan_kelas)
# =========================================================
print("="*70)
print("⚡ Memproses tabel 'catatan_kelas'")
print("="*70)

# 1. Ambil data mentah
df_catatan_lama = pd.read_sql("SELECT * FROM catatan_kelas", db_old)
print(f"Data mentah: {len(df_catatan_lama)} baris")

# 2. Simpan old_id untuk tracing
df_catatan_lama['old_id_ck'] = df_catatan_lama['idcatatan_kelas'].astype(str)
df_catatan_lama['old_id_jadwal'] = df_catatan_lama['idjadwal'].astype(str)

# Biarkan old_id_jadwal_detail apa adanya (bisa NaN)
# Jangan konversi ke string dulu agar null tetap null
df_catatan_lama['old_id_jadwal_detail'] = df_catatan_lama['idjadwaldetil']  # biarkan asli

# 3. Filter: hanya yang idjadwal-nya ada di mapping (parent jadwal valid)
df_catatan_lama = df_catatan_lama[df_catatan_lama['old_id_jadwal'].isin(mapping_id_jadwal.keys())].copy()
print(f"Setelah filter parent jadwal valid: {len(df_catatan_lama)} baris")

# 4. Buat dataframe sementara
df_temp = pd.DataFrame({
    'old_id_ck': df_catatan_lama['old_id_ck'],
    'old_id_jadwal': df_catatan_lama['old_id_jadwal'],
    'old_id_jadwal_detail': df_catatan_lama['old_id_jadwal_detail'],
    'catatan_kelas': df_catatan_lama['catatan'].fillna('').astype(str),
    'topik_diskusi': df_catatan_lama['materi_diskusi'].fillna('').astype(str),
    'hasil_konfirmasi': df_catatan_lama['hasil_konfirm'].fillna('').astype(str),
    'tanggal_konfirmasi': pd.to_datetime(df_catatan_lama['tglcek'], errors='coerce'),
})

# 5. Mapping id_jadwal
df_temp['id_jadwal'] = df_temp['old_id_jadwal'].map(mapping_id_jadwal)
df_temp = df_temp.dropna(subset=['id_jadwal'])

# 6. Mapping id_jadwal_detail (jika ada)
def map_detail(old_id):
    if pd.isna(old_id):
        return np.nan
    # old_id mungkin string atau angka, pastikan string untuk lookup di mapping
    return mapping_id_jadwal_detail.get(str(old_id), np.nan)

df_temp['id_jadwal_detail'] = df_temp['old_id_jadwal_detail'].apply(map_detail)

# 7. Hapus baris yang memiliki old_id_jadwal_detail tidak null tetapi mapping-nya gagal (orphan detail)
before_drop_detail = len(df_temp)
# Cek baris yang memiliki old_id_jadwal_detail tidak null dan id_jadwal_detail NaN
df_temp = df_temp[~((df_temp['old_id_jadwal_detail'].notna()) & (df_temp['id_jadwal_detail'].isna()))].copy()
print(f"Baris dengan detail orphan dibuang: {before_drop_detail - len(df_temp)}")

# 8. (Opsional) Pastikan detail yang tidak null ada di df_jadwal_detail_insert (inner join)
if len(df_jadwal_detail_insert) > 0:
    valid_detail_ids = set(df_jadwal_detail_insert['id'])  # internal ID
    df_temp = df_temp[~((df_temp['id_jadwal_detail'].notna()) & (~df_temp['id_jadwal_detail'].isin(valid_detail_ids)))].copy()
    print(f"Setelah filter detail dengan inner join: {len(df_temp)} baris")

# 9. Bersihkan tanggal_konfirmasi: jika null, isi dengan created_at (kita buat dulu)
df_temp['created_at'] = pd.Timestamp.now()
df_temp['tanggal_konfirmasi'] = df_temp['tanggal_konfirmasi'].fillna(df_temp['created_at'])

# 10. Tambahkan kolom id_karyawan (None)
df_temp['id_karyawan'] = None

# 11. Pilih kolom final (tanpa old_id)
df_catatan_kelas = df_temp[[
    'id_jadwal',
    'id_jadwal_detail',
    'catatan_kelas',
    'topik_diskusi',
    'tanggal_konfirmasi',
    'hasil_konfirmasi',
    'id_karyawan'
]].copy()

# 12. Tambahkan id (auto increment) internal
df_catatan_kelas.insert(0, 'id', range(1, len(df_catatan_kelas) + 1))

# 13. Buat mapping old_id_ck -> id baru (untuk keperluan jika ada anak)
mapping_id_catatan_kelas = dict(zip(df_temp['old_id_ck'], df_catatan_kelas['id']))

print(f"✅ catatan_kelas siap. Shape: {df_catatan_kelas.shape}")

# 14. Simpan ke fase_4_afrida
# 14. Simpan ke fase_4_afrida
fase_4_afrida['catatan_kelas'] = df_catatan_kelas
fase_4_afrida['mapping_id_catatan_kelas'] = mapping_id_catatan_kelas

⚡ Memproses tabel 'catatan_kelas'
Data mentah: 13733 baris
Setelah filter parent jadwal valid: 13733 baris
Baris dengan detail orphan dibuang: 0
Setelah filter detail dengan inner join: 13733 baris
✅ catatan_kelas siap. Shape: (13733, 8)


In [163]:
pk_jadwal = set(df_jadwal_insert['id'])
fk_jadwal = set(df_catatan_kelas['id_jadwal'])
invalid_jadwal = fk_jadwal - pk_jadwal

pk_detail = set(df_jadwal_detail_insert['id'])
fk_detail = set(df_catatan_kelas[df_catatan_kelas['id_jadwal_detail'].notna()]['id_jadwal_detail'])
invalid_detail = fk_detail - pk_detail

print(f"Invalid FK ke jadwal: {len(invalid_jadwal)}")
print(f"Invalid FK ke detail: {len(invalid_detail)}")
if len(invalid_jadwal)==0 and len(invalid_detail)==0:
    print("✅ Semua FK valid!")
else:
    print("❌ Ada invalid, periksa mapping.")

Invalid FK ke jadwal: 0
Invalid FK ke detail: 0
✅ Semua FK valid!


In [164]:
import pickle
import pandas as pd

# =========================================================
# 1. LOAD MAPPING TOPIK DISKUSI
# =========================================================
try:
    with open('mapping_id_topik.pkl', 'rb') as f:
        mapping_id_topik = pickle.load(f)
    print(f"✅ Mapping id_topik_diskusi di-load: {len(mapping_id_topik)} entri")
except FileNotFoundError:
    print("⚠️ File 'mapping_id_topik.pkl' tidak ditemukan. ID topik diskusi akan tetap string (berpotensi error).")
    mapping_id_topik = {}

# =========================================================
# 2. LOAD MAPPING CATATAN KELAS (SUDAH TERSEDIA)
# =========================================================
# Asumsikan mapping_id_catatan_kelas sudah tersedia dari fase 4.
# Jika tidak, bisa diambil dari fase_4_afrida.pkl
# Untuk amannya, coba load dari file jika belum ada:
try:
    mapping_id_catatan_kelas
except NameError:
    with open('fase_4_afrida.pkl', 'rb') as f:
        data = pickle.load(f)
    mapping_id_catatan_kelas = data.get('mapping_id_catatan_kelas', {})
    print(f"✅ Mapping id_catatan_kelas di-load: {len(mapping_id_catatan_kelas)} entri")

# =========================================================
# 3. PROSES DATA catatan_kelas_tag
# =========================================================
print("\n=== Proses catatan_kelas_tag ===")

# Ambil data mentah dari DB lama
df_tag_lama = pd.read_sql("SELECT * FROM catatan_kelas_tag", db_old)
print(f"Data mentah: {len(df_tag_lama)} baris")

# Simpan old_id_ck
df_tag_lama['old_id_ck'] = df_tag_lama['idcatatan_kelas'].astype(str)

# Filter: hanya yang old_id_ck-nya ada di mapping catatan_kelas
df_tag_lama = df_tag_lama[df_tag_lama['old_id_ck'].isin(mapping_id_catatan_kelas.keys())].copy()
print(f"Setelah filter parent catatan_kelas valid: {len(df_tag_lama)} baris")

# Buat dataframe sementara
df_temp = pd.DataFrame({
    'old_id_ck': df_tag_lama['old_id_ck'],
    'id_topik_diskusi_old': df_tag_lama['idtagmd'].astype(str),
})

# 3a. Mapping id_catatan_kelas
df_temp['id_ck'] = df_temp['old_id_ck'].map(mapping_id_catatan_kelas)
df_temp = df_temp.dropna(subset=['id_ck'])
df_temp['id_ck'] = df_temp['id_ck'].astype('int64')

# 3b. Mapping id_topik_diskusi (jika mapping tersedia)
if mapping_id_topik:
    df_temp['id_topik_diskusi'] = df_temp['id_topik_diskusi_old'].map(mapping_id_topik)
    before = len(df_temp)
    df_temp = df_temp.dropna(subset=['id_topik_diskusi'])
    print(f"  Baris tanpa mapping topik diskusi dibuang: {before - len(df_temp)}")
    df_temp['id_topik_diskusi'] = df_temp['id_topik_diskusi'].astype('int64')
else:
    # Jika tidak ada mapping, biarkan sebagai string (akan error saat insert)
    df_temp['id_topik_diskusi'] = df_temp['id_topik_diskusi_old']
    print("  ⚠️ ID topik diskusi tidak di-mapping, akan tetap string.")

# Hapus kolom bantu
df_temp.drop(columns=['old_id_ck', 'id_topik_diskusi_old'], inplace=True)

# 3c. (Opsional) Inner join dengan catatan_kelas untuk memastikan FK valid
pk_ck = set(df_catatan_kelas['id'])  # internal PK catatan_kelas
df_temp = df_temp[df_temp['id_ck'].isin(pk_ck)].copy()

# 4. Tambahkan id internal (auto increment)
df_temp.insert(0, 'id', range(1, len(df_temp) + 1))

# 5. Dataframe final (kolom: id, id_ck, id_topik_diskusi)
df_catatan_kelas_tag = df_temp[['id', 'id_ck', 'id_topik_diskusi']].copy()

print(f"✅ catatan_kelas_tag siap. Shape: {df_catatan_kelas_tag.shape}")

# 6. Simpan ke fase_4_afrida
fase_4_afrida['catatan_kelas_tag'] = df_catatan_kelas_tag

✅ Mapping id_topik_diskusi di-load: 11 entri

=== Proses catatan_kelas_tag ===
Data mentah: 1067 baris
Setelah filter parent catatan_kelas valid: 1067 baris
  Baris tanpa mapping topik diskusi dibuang: 0
✅ catatan_kelas_tag siap. Shape: (1067, 3)


In [165]:
import pickle
import pandas as pd

# Load mapping topik
with open('mapping_id_topik.pkl', 'rb') as f:
    mapping_id_topik = pickle.load(f)

# Ambil data mentah tag
df_check = pd.read_sql("SELECT idcatatan_kelas, idtagmd FROM catatan_kelas_tag", db_old)
df_check['old_id_topik'] = df_check['idtagmd'].astype(str)

# Cek apakah ada yang tidak ada di mapping
df_check['ada_di_mapping'] = df_check['old_id_topik'].isin(mapping_id_topik.keys())
total = len(df_check)
tidak_cocok = df_check[~df_check['ada_di_mapping']]

print(f"Total data tag: {total}")
print(f"ID topik yang cocok: {df_check['ada_di_mapping'].sum()}")
print(f"ID topik TIDAK cocok: {len(tidak_cocok)}")

if len(tidak_cocok) > 0:
    print("\n🔴 Contoh ID topik yang tidak cocok:")
    print(tidak_cocok['old_id_topik'].unique()[:10])
    
    # Tampilkan beberapa baris contoh
    print("\nContoh data yang tidak cocok:")
    display(tidak_cocok.head())
else:
    print("✅ Semua id_topik_diskusi cocok dengan mapping!")

Total data tag: 1067
ID topik yang cocok: 1067
ID topik TIDAK cocok: 0
✅ Semua id_topik_diskusi cocok dengan mapping!


In [166]:
pk_ck = set(df_catatan_kelas['id'])
fk_tag = set(df_catatan_kelas_tag['id_ck'])
invalid = fk_tag - pk_ck
print(f"Invalid FK ke catatan_kelas: {len(invalid)}")
print("Aman! ✅" if len(invalid)==0 else "❌")

Invalid FK ke catatan_kelas: 0
Aman! ✅


In [167]:
# =========================================================
# 9. TRANSFORMASI TABEL: catatan_mingguan (sumber: catatan_mingguan)
# =========================================================
print("="*70)
print("⚡ Memproses tabel 'catatan_mingguan'")
print("="*70)

# 1. Ambil data mentah
df_cm_lama = pd.read_sql("SELECT * FROM catatan_mingguan", db_old)
print(f"Data mentah: {len(df_cm_lama)} baris")

# 2. Simpan old_id untuk mapping (jika dibutuhkan nanti)
df_cm_lama['old_id_cm'] = df_cm_lama['idcatatanweek'].astype(str)

# 3. Buat dataframe sementara
df_temp = pd.DataFrame({
    'old_id_cm': df_cm_lama['old_id_cm'],
    'id_user': df_cm_lama['idusers'].astype(str),
    'tanggal_mulai_cm': pd.to_datetime(df_cm_lama['tglawal'], errors='coerce').dt.date,
    'tanggal_selesai_cm': pd.to_datetime(df_cm_lama['tglakhir'], errors='coerce').dt.date,
    'tanggal_verifikasi_cm': pd.to_datetime(df_cm_lama['tglcek'], errors='coerce'),
    'keterangan_cm': df_cm_lama['catatan'].fillna('').astype(str),
    'keputusan_cm': df_cm_lama['hasil_konfirmasi'].fillna('').astype(str),
})

# 4. Hapus baris yang tidak punya id_user (FK ke users)
before_drop_user = len(df_temp)
df_temp = df_temp.dropna(subset=['id_user'])
print(f"Baris tanpa id_user dibuang: {before_drop_user - len(df_temp)}")

# 5. Tambahkan kolom created_at & updated_at
df_temp['created_at'] = pd.Timestamp.now()
df_temp['updated_at'] = pd.Timestamp.now()

# 6. (Opsional) Jika ada kolom lain yang perlu ditambahkan default, lakukan di sini
# Misalnya kolom 'status' atau 'jenis' jika ada di skema baru

# 7. Pilih kolom final (tanpa old_id)
df_catatan_mingguan = df_temp[[
    'id_user',
    'tanggal_mulai_cm',
    'tanggal_selesai_cm',
    'tanggal_verifikasi_cm',
    'keterangan_cm',
    'keputusan_cm',
    'created_at',
    'updated_at'
]].copy()

# 8. Tambahkan id (auto increment) internal
df_catatan_mingguan.insert(0, 'id', range(1, len(df_catatan_mingguan) + 1))

# 9. Buat mapping old_id_cm -> id baru (untuk keperluan jika ada tabel anak)
mapping_id_catatan_mingguan = dict(zip(df_temp['old_id_cm'], df_catatan_mingguan['id']))

print(f"✅ catatan_mingguan siap. Shape: {df_catatan_mingguan.shape}")

# 10. Simpan ke fase_4_afrida
fase_4_afrida['catatan_mingguan'] = df_catatan_mingguan
fase_4_afrida['mapping_id_catatan_mingguan'] = mapping_id_catatan_mingguan

⚡ Memproses tabel 'catatan_mingguan'
Data mentah: 0 baris
Baris tanpa id_user dibuang: 0
✅ catatan_mingguan siap. Shape: (0, 9)


In [168]:
# =========================================================
# PERBAIKAN AKHIR SEBELUM SIMPAN PICKLE CLEAN
# =========================================================
print("\n🔧 Melakukan perbaikan pada tabel bermasalah...")

# 1. PERBAIKAN jadwal_pengajar: hapus kolom created_at & updated_at
if 'jadwal_pengajar' in fase_4_afrida:
    df = fase_4_afrida['jadwal_pengajar']
    for col in ['created_at', 'updated_at']:
        if col in df.columns:
            df = df.drop(columns=[col])
            print(f"  ✔ {col} dihapus dari jadwal_pengajar")
    fase_4_afrida['jadwal_pengajar'] = df

# 2. PERBAIKAN jadwal_siswa: isi tanggal null dengan default (sekarang)
if 'jadwal_siswa' in fase_4_afrida:
    df = fase_4_afrida['jadwal_siswa']
    now = pd.Timestamp.now()  # nilai default
    
    # Kolom yang wajib diisi (NOT NULL)
    for col in ['tanggal_mulai', 'tanggal_aktif']:
        if col in df.columns:
            df[col] = df[col].fillna(now)
            print(f"  ✔ {col} diisi dengan '{now}' untuk yang null")
    
    # Kolom yang boleh NULL (tapi kita isi dengan NULL agar tidak error)
    if 'tanggal_keluar' in df.columns:
        df['tanggal_keluar'] = df['tanggal_keluar'].fillna(pd.NA)
        print(f"  ✔ tanggal_keluar diisi NULL (pd.NA) untuk yang null")
    
    # Pastikan tipe datetime (biarkan apa adanya)
    fase_4_afrida['jadwal_siswa'] = df

print("✅ Perbaikan selesai.\n")


🔧 Melakukan perbaikan pada tabel bermasalah...
  ✔ created_at dihapus dari jadwal_pengajar
  ✔ updated_at dihapus dari jadwal_pengajar
✅ Perbaikan selesai.



In [169]:
# =========================================================
# PERBAIKAN AKHIR UNTUK MENGATASI ERROR INSERT
# =========================================================
print("\n🔧 Melakukan perbaikan akhir sebelum pickle...")

# 2. Perbaiki jadwal_pengajar: set id_user menjadi NULL karena belum ada mapping users
if 'jadwal_pengajar' in fase_4_afrida:
    df = fase_4_afrida['jadwal_pengajar']
    for col in ['created_at', 'updated_at']:
        if col in df.columns:
            df = df.drop(columns=[col])
    fase_4_afrida['jadwal_pengajar'] = df
    print("  ✔ jadwal_pengajar: id_user di-set NULL, kolom created_at/updated_at dihapus")

# 3. Perbaiki catatan_kelas_tag: ganti kolom 'id_catatan_kelas' menjadi 'id_ck' sesuai skema
if 'catatan_kelas_tag' in fase_4_afrida:
    df = fase_4_afrida['catatan_kelas_tag']
    if 'id_catatan_kelas' in df.columns:
        df = df.rename(columns={'id_catatan_kelas': 'id_ck'})
        fase_4_afrida['catatan_kelas_tag'] = df
        print("  ✔ catatan_kelas_tag: kolom 'id_catatan_kelas' di-rename menjadi 'id_ck'")
    else:
        print("  ⚠ catatan_kelas_tag: kolom 'id_catatan_kelas' tidak ditemukan, cek struktur")

# 4. Pastikan tidak ada kolom 'created_at' di jadwal_detail (jika ada) karena error timestamp
if 'jadwal_detail' in fase_4_afrida:
    df = fase_4_afrida['jadwal_detail']
    # Hapus kolom created_at, updated_at, last_generated_at jika ada dan tidak diperlukan
    for col in ['created_at', 'updated_at', 'last_generated_at']:
        if col in df.columns:
            df = df.drop(columns=[col])
            print(f"  ✔ jadwal_detail: {col} dihapus")
    fase_4_afrida['jadwal_detail'] = df

# 5. Perbaiki jadwal_siswa: isi tanggal yang null dengan NULL (boleh null setelah ubah struktur)
#    Tapi kita sudah ubah struktur di db_future, jadi biarkan NULL
#    Pastikan kolom datetime sudah di-fix
if 'jadwal_siswa' in fase_4_afrida:
    df = fase_4_afrida['jadwal_siswa']
    # Pastikan tidak ada kolom created_at, updated_at (jika ada hapus)
    for col in ['created_at', 'updated_at']:
        if col in df.columns:
            df = df.drop(columns=[col])
            print(f"  ✔ jadwal_siswa: {col} dihapus")
    fase_4_afrida['jadwal_siswa'] = df

# 6. Perbaiki catatan_kelas: hapus kolom created_at/updated_at jika ada
if 'catatan_kelas' in fase_4_afrida:
    df = fase_4_afrida['catatan_kelas']
    for col in ['created_at', 'updated_at']:
        if col in df.columns:
            df = df.drop(columns=[col])
            print(f"  ✔ catatan_kelas: {col} dihapus")
    fase_4_afrida['catatan_kelas'] = df

print("✅ Perbaikan selesai.")


🔧 Melakukan perbaikan akhir sebelum pickle...
  ✔ jadwal_pengajar: id_user di-set NULL, kolom created_at/updated_at dihapus
  ⚠ catatan_kelas_tag: kolom 'id_catatan_kelas' tidak ditemukan, cek struktur
  ✔ jadwal_detail: created_at dihapus
  ✔ jadwal_detail: updated_at dihapus
  ✔ jadwal_detail: last_generated_at dihapus
✅ Perbaikan selesai.


In [170]:
import pickle
import pandas as pd

# =========================================================
# 1. LOAD FILE PICKLE
# =========================================================
print("="*70)
print("🧹 Membersihkan kolom 'id' dari semua DataFrame")
print("="*70)

# Load file pickle
with open('fase_4_afrida.pkl', 'rb') as f:
    fase_4_afrida = pickle.load(f)

print(f"📋 Key yang tersedia: {list(fase_4_afrida.keys())}")

# =========================================================
# 2. DAFTAR TABEL YANG PERLU DIBERSIHKAN
# =========================================================
tabel_list = [
    'jadwal',
    'jadwal_hari',
    'jadwal_detail',
    'jadwal_pengajar',
    'jadwal_siswa',
    'catatan_kelas',
    'catatan_kelas_tag',
    'catatan_mingguan',
    'presensi_siswa',
    'catatan_siswa',
    'followup_cs'
]

# =========================================================
# 3. HAPUS KOLOM 'id' DARI SETIAP DATAFRAME
# =========================================================
print("\n📌 Menghapus kolom 'id'...")
print("-"*50)

for tbl in tabel_list:
    if tbl in fase_4_afrida and isinstance(fase_4_afrida[tbl], pd.DataFrame):
        df = fase_4_afrida[tbl]
        before_cols = df.columns.tolist()
        
        if 'id' in df.columns:
            fase_4_afrida[tbl] = df.drop(columns=['id'])
            print(f"  ✅ {tbl}: kolom 'id' dihapus")
            print(f"     Sebelum: {before_cols}")
            print(f"     Sesudah: {fase_4_afrida[tbl].columns.tolist()}")
        else:
            print(f"  ℹ️ {tbl}: tidak ada kolom 'id'")
            print(f"     Kolom: {before_cols}")
    else:
        print(f"  ⚠️ {tbl}: tidak ditemukan di pickle")

# =========================================================
# 4. VERIFIKASI: PASTIKAN TIDAK ADA KOLOM 'id'
# =========================================================
print("\n" + "="*70)
print("🔍 VERIFIKASI SETELAH PEMBERSIHAN")
print("="*70)

masih_ada_id = []
for tbl in tabel_list:
    if tbl in fase_4_afrida and isinstance(fase_4_afrida[tbl], pd.DataFrame):
        if 'id' in fase_4_afrida[tbl].columns:
            masih_ada_id.append(tbl)
            print(f"  ❌ {tbl}: MASIH ADA kolom 'id'")
        else:
            print(f"  ✅ {tbl}: bersih (tidak ada kolom 'id')")

if masih_ada_id:
    print(f"\n⚠️ PERHATIAN: kolom 'id' masih ada di: {masih_ada_id}")
    print("   Periksa kembali proses transformasi tabel tersebut.")
else:
    print("\n✅ SEMUA tabel sudah bersih dari kolom 'id'!")


🧹 Membersihkan kolom 'id' dari semua DataFrame
📋 Key yang tersedia: ['jadwal', 'jadwal_old_ids', 'jadwal_hari', 'mapping_id_jadwal', 'jadwal_detail', 'jadwal_detail_old_ids', 'mapping_id_jadwal_detail', 'jadwal_pengajar', 'catatan_kelas_tag', 'mapping_id_jadwal_siswa', 'catatan_kelas', 'mapping_id_catatan_kelas', 'catatan_mingguan', 'mapping_id_catatan_mingguan', 'jadwal_siswa']

📌 Menghapus kolom 'id'...
--------------------------------------------------
  ℹ️ jadwal: tidak ada kolom 'id'
     Kolom: ['id_kursus', 'id_periode', 'id_level', 'id_sesi', 'metode_belajar_jadwal', 'nama_rombel', 'status_arsip', 'tempat']
  ℹ️ jadwal_hari: tidak ada kolom 'id'
     Kolom: ['id_jadwal', 'nama_hari']
  ℹ️ jadwal_detail: tidak ada kolom 'id'
     Kolom: ['judul', 'deskripsi', 'url_jadwal_detail', 'id_jadwal', 'label_warna', 'penanda_mulai', 'penanda_selesai', 'id_mitra', 'id_sesi_override', 'status_detail', 'source_type', 'original_jadwal_detail_id', 'has_operational_data']
  ℹ️ jadwal_pengajar:

In [171]:
import pickle

# =========================================================
# SIMPAN df_jadwal_siswa KE fase_4_afrida.pkl
# =========================================================
print("="*70)
print("💾 Menyimpan df_jadwal_siswa ke fase_4_afrida.pkl")
print("="*70)

# 1. Load file pickle yang ada
try:
    with open('fase_4_afrida.pkl', 'rb') as f:
        fase_4_afrida = pickle.load(f)
    print("✅ fase_4_afrida.pkl ditemukan, melanjutkan...")
except FileNotFoundError:
    fase_4_afrida = {}
    print("📂 fase_4_afrida.pkl belum ada, membuat baru...")

# 2. Tambahkan df_jadwal_siswa ke dictionary
if 'df_jadwal_siswa' in locals():
    fase_4_afrida['jadwal_siswa'] = df_jadwal_siswa
    print(f"✅ df_jadwal_siswa ditambahkan ke fase_4_afrida")
    print(f"   Shape: {df_jadwal_siswa.shape}")
    print(f"   Kolom: {df_jadwal_siswa.columns.tolist()}")
else:
    print("❌ df_jadwal_siswa tidak ditemukan di memory!")
    print("   Pastikan df_jadwal_siswa sudah dibuat sebelumnya.")

# 3. Simpan mapping (jika ada)
if 'mapping_id_jadwal_siswa' in locals():
    fase_4_afrida['mapping_id_jadwal_siswa'] = mapping_id_jadwal_siswa
    print(f"✅ mapping_id_jadwal_siswa ditambahkan: {len(mapping_id_jadwal_siswa)} entri")

# 4. Simpan ke file
with open('fase_4_afrida.pkl', 'wb') as f:
    pickle.dump(data_fase4, f)

print(f"\n✅ Data berhasil disimpan ke fase_4_afrida.pkl")
print(f"   Key yang tersedia: {list(data_fase4.keys())}")

# 5. Verifikasi
print("\n🔍 Verifikasi:")
with open('fase_4_afrida.pkl', 'rb') as f:
    verify_data = pickle.load(f)
    if 'jadwal_siswa' in verify_data:
        print(f"  ✅ jadwal_siswa: {verify_data['jadwal_siswa'].shape}")
    else:
        print("  ❌ jadwal_siswa tidak ditemukan setelah disimpan")

💾 Menyimpan df_jadwal_siswa ke fase_4_afrida.pkl
✅ fase_4_afrida.pkl ditemukan, melanjutkan...
✅ df_jadwal_siswa ditambahkan ke fase_4_afrida
   Shape: (3958, 15)
   Kolom: ['id', 'id_jadwal', 'id_siswa', 'tanggal_mulai', 'tanggal_keluar', 'tanggal_aktif', 'tambahan_sesi', 'tambahan_keterangan', 'status_keluar', 'is_acc_rapor', 'status_ketuntasan', 'catatan_ketuntasan_guru', 'catatan_ketuntasan_admin', 'ketuntasan_diperbarui_oleh', 'ketuntasan_diperbarui_pada']
✅ mapping_id_jadwal_siswa ditambahkan: 3958 entri

✅ Data berhasil disimpan ke fase_4_afrida.pkl
   Key yang tersedia: ['jadwal', 'jadwal_old_ids', 'jadwal_hari', 'mapping_id_jadwal', 'jadwal_detail', 'jadwal_detail_old_ids', 'mapping_id_jadwal_detail', 'jadwal_pengajar', 'catatan_kelas_tag', 'mapping_id_jadwal_siswa', 'catatan_kelas', 'mapping_id_catatan_kelas', 'catatan_mingguan', 'mapping_id_catatan_mingguan', 'jadwal_siswa']

🔍 Verifikasi:
  ✅ jadwal_siswa: (3958, 14)


In [172]:
import pickle
import pandas as pd

# =========================================================
# 1. LOAD FILE PICKLE
# =========================================================
print("="*70)
print("🧹 Membersihkan kolom 'id' dari semua DataFrame")
print("="*70)

# Load file pickle
with open('fase_4_afrida.pkl', 'rb') as f:
    fase_4_afrida = pickle.load(f)

print(f"📋 Key yang tersedia: {list(fase_4_afrida.keys())}")

# =========================================================
# 2. DAFTAR TABEL YANG PERLU DIBERSIHKAN
# =========================================================
tabel_list = [
    'jadwal',
    'jadwal_hari',
    'jadwal_detail',
    'jadwal_pengajar',
    'jadwal_siswa',
    'catatan_kelas',
    'catatan_kelas_tag',
    'catatan_mingguan',
    'presensi_siswa',
    'catatan_siswa',
    'followup_cs'
]

# =========================================================
# 3. HAPUS KOLOM 'id' DARI SETIAP DATAFRAME
# =========================================================
print("\n📌 Menghapus kolom 'id'...")
print("-"*50)

for tbl in tabel_list:
    if tbl in fase_4_afrida and isinstance(fase_4_afrida[tbl], pd.DataFrame):
        df = fase_4_afrida[tbl]
        before_cols = df.columns.tolist()
        
        if 'id' in df.columns:
            fase_4_afrida[tbl] = df.drop(columns=['id'])
            print(f"  ✅ {tbl}: kolom 'id' dihapus")
            print(f"     Sebelum: {before_cols}")
            print(f"     Sesudah: {fase_4_afrida[tbl].columns.tolist()}")
        else:
            print(f"  ℹ️ {tbl}: tidak ada kolom 'id'")
            print(f"     Kolom: {before_cols}")
    else:
        print(f"  ⚠️ {tbl}: tidak ditemukan di pickle")

# =========================================================
# 4. VERIFIKASI: PASTIKAN TIDAK ADA KOLOM 'id'
# =========================================================
print("\n" + "="*70)
print("🔍 VERIFIKASI SETELAH PEMBERSIHAN")
print("="*70)

masih_ada_id = []
for tbl in tabel_list:
    if tbl in fase_4_afrida and isinstance(fase_4_afrida[tbl], pd.DataFrame):
        if 'id' in fase_4_afrida[tbl].columns:
            masih_ada_id.append(tbl)
            print(f"  ❌ {tbl}: MASIH ADA kolom 'id'")
        else:
            print(f"  ✅ {tbl}: bersih (tidak ada kolom 'id')")

if masih_ada_id:
    print(f"\n⚠️ PERHATIAN: kolom 'id' masih ada di: {masih_ada_id}")
    print("   Periksa kembali proses transformasi tabel tersebut.")
else:
    print("\n✅ SEMUA tabel sudah bersih dari kolom 'id'!")

# =========================================================
# 5. SIMPAN ULANG KE FILE PICKLE (CLEAN)
# =========================================================
print("\n" + "="*70)
print("💾 Menyimpan ulang ke fase_4_afrida.pkl (clean)")
print("="*70)

with open('fase_4_afrida.pkl', 'wb') as f:
    pickle.dump(fase_4_afrida, f)

print("✅ File 'fase_4_afrida.pkl' berhasil disimpan (clean)")

# =========================================================
# 6. RINGKASAN AKHIR
# =========================================================
print("\n" + "="*70)
print("📊 RINGKASAN AKHIR")
print("="*70)

for tbl in tabel_list:
    if tbl in fase_4_afrida and isinstance(fase_4_afrida[tbl], pd.DataFrame):
        df = fase_4_afrida[tbl]
        print(f"  {tbl:<20} : {df.shape[0]:>6} baris | {df.shape[1]:>3} kolom")
        print(f"                   Kolom: {df.columns.tolist()}")

print("\n✅ Selesai!")

🧹 Membersihkan kolom 'id' dari semua DataFrame
📋 Key yang tersedia: ['jadwal', 'jadwal_old_ids', 'jadwal_hari', 'mapping_id_jadwal', 'jadwal_detail', 'jadwal_detail_old_ids', 'mapping_id_jadwal_detail', 'jadwal_pengajar', 'catatan_kelas_tag', 'mapping_id_jadwal_siswa', 'catatan_kelas', 'mapping_id_catatan_kelas', 'catatan_mingguan', 'mapping_id_catatan_mingguan', 'jadwal_siswa']

📌 Menghapus kolom 'id'...
--------------------------------------------------
  ℹ️ jadwal: tidak ada kolom 'id'
     Kolom: ['id_kursus', 'id_periode', 'id_level', 'id_sesi', 'metode_belajar_jadwal', 'nama_rombel', 'status_arsip', 'tempat']
  ℹ️ jadwal_hari: tidak ada kolom 'id'
     Kolom: ['id_jadwal', 'nama_hari']
  ℹ️ jadwal_detail: tidak ada kolom 'id'
     Kolom: ['judul', 'deskripsi', 'url_jadwal_detail', 'id_jadwal', 'label_warna', 'penanda_mulai', 'penanda_selesai', 'id_mitra', 'id_sesi_override', 'status_detail', 'source_type', 'original_jadwal_detail_id', 'has_operational_data']
  ℹ️ jadwal_pengajar:

In [173]:
import pickle
import pandas as pd

# =========================================================
# 1. Tentukan file pickle yang akan di-load
# =========================================================
file_pkl = 'fase_4_afrida.pkl'   # Ganti dengan nama file yang sesuai

try:
    with open(file_pkl, 'rb') as f:
        data = pickle.load(f)
    print(f"✅ Berhasil load {file_pkl}")
    print(f"📌 Key yang tersedia: {list(data.keys())}\n")
except FileNotFoundError:
    print(f"❌ File {file_pkl} tidak ditemukan. Coba gunakan 'fase_4_afrida.pkl'")
    # Jika file tidak ditemukan, coba alternatif
    file_pkl = 'fase_4_afrida.pkl'
    try:
        with open(file_pkl, 'rb') as f:
            data = pickle.load(f)
        print(f"✅ Berhasil load {file_pkl}")
        print(f"📌 Key yang tersedia: {list(data.keys())}\n")
    except FileNotFoundError:
        print("❌ Tidak ada file pickle yang ditemukan.")
        exit()

# =========================================================
# 2. Tampilkan informasi setiap tabel (DataFrame)
# =========================================================
print("="*80)
print("📊 INFORMASI TABEL DI DALAM PICKLE")
print("="*80)

# Filter key yang merupakan DataFrame
df_keys = [k for k, v in data.items() if isinstance(v, pd.DataFrame)]
print(f"\n📋 Jumlah tabel: {len(df_keys)}")

for key in df_keys:
    df = data[key]
    print(f"\n🔹 Nama tabel : {key}")
    print(f"   Shape      : {df.shape}")
    print(f"   Kolom      : {list(df.columns)}")
    print(f"   Jumlah null per kolom:")
    print(df.isnull().sum().to_string())
    print(f"\n   Contoh data (5 baris pertama):")
    display(df.head())   # Jika di Jupyter/Colab
    # Jika tidak pakai display, ganti dengan print(df.head())
    print("-"*80)

# =========================================================
# 3. (Opsional) Tampilkan informasi mapping (jika ada)
# =========================================================
print("\n📌 INFORMASI MAPPING (jika ada):")
mapping_keys = [k for k in data.keys() if k.startswith('mapping_')]
for key in mapping_keys:
    mapping = data[key]
    if isinstance(mapping, dict):
        print(f"   {key} : {len(mapping)} entri")
        # Tampilkan 5 contoh mapping pertama
        contoh = list(mapping.items())[:5]
        print(f"     Contoh: {contoh}")
    else:
        print(f"   {key} : {type(mapping)}")

✅ Berhasil load fase_4_afrida.pkl
📌 Key yang tersedia: ['jadwal', 'jadwal_old_ids', 'jadwal_hari', 'mapping_id_jadwal', 'jadwal_detail', 'jadwal_detail_old_ids', 'mapping_id_jadwal_detail', 'jadwal_pengajar', 'catatan_kelas_tag', 'mapping_id_jadwal_siswa', 'catatan_kelas', 'mapping_id_catatan_kelas', 'catatan_mingguan', 'mapping_id_catatan_mingguan', 'jadwal_siswa']

📊 INFORMASI TABEL DI DALAM PICKLE

📋 Jumlah tabel: 8

🔹 Nama tabel : jadwal
   Shape      : (556, 8)
   Kolom      : ['id_kursus', 'id_periode', 'id_level', 'id_sesi', 'metode_belajar_jadwal', 'nama_rombel', 'status_arsip', 'tempat']
   Jumlah null per kolom:
id_kursus                0
id_periode               0
id_level                 0
id_sesi                  0
metode_belajar_jadwal    0
nama_rombel              0
status_arsip             0
tempat                   0

   Contoh data (5 baris pertama):


,id_kursus,id_periode,id_level,id_sesi,metode_belajar_jadwal,nama_rombel,status_arsip,tempat
0,K00001,P00006,L00017,S00002,Online,01 GOGO 3B SR2 (ERICA),1,Ruang Kelas 4
1,K00001,P00006,L00024,S00002,Offline,02 SO 1C SR2 (QORIN),1,Ruang Kelas 5
2,K00001,P00006,L00023,S00001,Offline,03 SO 1B SR1 (TATIK),1,Ruang Kelas 1
3,K00001,P00006,L00025,S00003,Offline,04 SO 2A SR3 (TATIK),1,Ruang Kelas 1
4,K00001,P00006,L00014,S00003,Offline,05 GOGO 1B SelK3 (ERICA),1,Ruang Kelas 4


--------------------------------------------------------------------------------

🔹 Nama tabel : jadwal_hari
   Shape      : (982, 2)
   Kolom      : ['id_jadwal', 'nama_hari']
   Jumlah null per kolom:
id_jadwal    0
nama_hari    0

   Contoh data (5 baris pertama):


,id_jadwal,nama_hari
0,1,Senin
1,1,Rabu
2,2,Senin
3,2,Rabu
4,3,Senin


--------------------------------------------------------------------------------

🔹 Nama tabel : jadwal_detail
   Shape      : (17312, 13)
   Kolom      : ['judul', 'deskripsi', 'url_jadwal_detail', 'id_jadwal', 'label_warna', 'penanda_mulai', 'penanda_selesai', 'id_mitra', 'id_sesi_override', 'status_detail', 'source_type', 'original_jadwal_detail_id', 'has_operational_data']
   Jumlah null per kolom:
judul                            0
deskripsi                        0
url_jadwal_detail                0
id_jadwal                        0
label_warna                      0
penanda_mulai                    0
penanda_selesai                  0
id_mitra                     17312
id_sesi_override             17312
status_detail                    0
source_type                      0
original_jadwal_detail_id    17312
has_operational_data             0

   Contoh data (5 baris pertama):


...

--------------------------------------------------------------------------------

🔹 Nama tabel : catatan_mingguan
   Shape      : (0, 8)
   Kolom      : ['id_user', 'tanggal_mulai_cm', 'tanggal_selesai_cm', 'tanggal_verifikasi_cm', 'keterangan_cm', 'keputusan_cm', 'created_at', 'updated_at']
   Jumlah null per kolom:
id_user                  0
tanggal_mulai_cm         0
tanggal_selesai_cm       0
tanggal_verifikasi_cm    0
keterangan_cm            0
keputusan_cm             0
created_at               0
updated_at               0

   Contoh data (5 baris pertama):


,id_user,tanggal_mulai_cm,tanggal_selesai_cm,tanggal_verifikasi_cm,keterangan_cm,keputusan_cm,created_at,updated_at


--------------------------------------------------------------------------------

🔹 Nama tabel : jadwal_siswa
   Shape      : (3958, 14)
   Kolom      : ['id_jadwal', 'id_siswa', 'tanggal_mulai', 'tanggal_keluar', 'tanggal_aktif', 'tambahan_sesi', 'tambahan_keterangan', 'status_keluar', 'is_acc_rapor', 'status_ketuntasan', 'catatan_ketuntasan_guru', 'catatan_ketuntasan_admin', 'ketuntasan_diperbarui_oleh', 'ketuntasan_diperbarui_pada']
   Jumlah null per kolom:
id_jadwal                        0
id_siswa                         0
tanggal_mulai                 2855
tanggal_keluar                3922
tanggal_aktif                 3958
tambahan_sesi                    0
tambahan_keterangan              0
status_keluar                    0
is_acc_rapor                     0
status_ketuntasan             3958
catatan_ketuntasan_guru       3958
catatan_ketuntasan_admin      3958
ketuntasan_diperbarui_oleh    3958
ketuntasan_diperbarui_pada    3958

   Contoh data (5 baris pertama):


,id_jadwal,id_siswa,tanggal_mulai,tanggal_keluar,tanggal_aktif,tambahan_sesi,tambahan_keterangan,status_keluar,is_acc_rapor,status_ketuntasan,catatan_ketuntasan_guru,catatan_ketuntasan_admin,ketuntasan_diperbarui_oleh,ketuntasan_diperbarui_pada
0,3,347,NaT,NaT,NaT,0,Belum ada keterangan,0,0,None,None,None,None,None
1,3,348,NaT,NaT,NaT,0,Belum ada keterangan,0,0,None,None,None,None,None
2,7,79,NaT,NaT,NaT,0,Belum ada keterangan,0,0,None,None,None,None,None
3,7,82,NaT,NaT,NaT,0,Belum ada keterangan,0,0,None,None,None,None,None
4,7,108,NaT,NaT,NaT,0,Belum ada keterangan,0,0,None,None,None,None,None


--------------------------------------------------------------------------------

📌 INFORMASI MAPPING (jika ada):
   mapping_id_jadwal : 556 entri
     Contoh: [('J000000023', 1), ('J000000024', 2), ('J000000025', 3), ('J000000026', 4), ('J000000027', 5)]
   mapping_id_jadwal_detail : 17312 entri
     Contoh: [('D0000000000000002497', 1), ('D0000000000000002498', 2), ('D0000000000000002499', 3), ('D0000000000000002500', 4), ('D0000000000000002501', 5)]
   mapping_id_jadwal_siswa : 3958 entri
     Contoh: [('P0000069', 1), ('P0000070', 2), ('P0000072', 3), ('P0000073', 4), ('P0000074', 5)]
   mapping_id_catatan_kelas : 13733 entri
     Contoh: [('C00001', 1), ('C00002', 2), ('C00003', 3), ('C00004', 4), ('C00005', 5)]
   mapping_id_catatan_mingguan : 0 entri
     Contoh: []


In [174]:
import pandas as pd
import pickle

# =========================================================
# 1. LOAD SEMUA MAPPING YANG DIBUTUHKAN
# =========================================================
print("="*70)
print("🔍 VALIDASI MAPPING & FOREIGN KEY")
print("="*70)

# Load mapping dari fase_4_afrida.pkl
print("\n📂 Memuat fase_4_afrida.pkl...")
with open('fase_4_afrida.pkl', 'rb') as f:
    data = pickle.load(f)

# Ambil semua mapping yang tersedia
mapping_id_jadwal = data.get('mapping_id_jadwal', {})
mapping_id_jadwal_detail = data.get('mapping_id_jadwal_detail', {})
mapping_id_catatan_kelas = data.get('mapping_id_catatan_kelas', {})
mapping_id_jadwal_siswa = data.get('mapping_id_jadwal_siswa', {})
mapping_id_siswa = data.get('mapping_id_siswa', {})

# Load mapping_siswa.pkl (jika ada file terpisah)
try:
    with open('mapping_siswa.pkl', 'rb') as f:
        mapping_siswa_df = pickle.load(f)
    if isinstance(mapping_siswa_df, pd.DataFrame):
        mapping_id_siswa = dict(zip(
            mapping_siswa_df[mapping_siswa_df.columns[0]].astype(str),
            mapping_siswa_df[mapping_siswa_df.columns[1]]
        ))
    elif isinstance(mapping_siswa_df, dict):
        mapping_id_siswa = mapping_siswa_df
    print(f"✅ Mapping siswa dari mapping_siswa.pkl: {len(mapping_id_siswa)} entri")
except FileNotFoundError:
    print("⚠️ mapping_siswa.pkl tidak ditemukan, gunakan yang ada di fase_4_afrida.pkl")
except Exception as e:
    print(f"⚠️ Gagal load mapping_siswa.pkl: {e}")

print(f"\n📊 RINGKASAN MAPPING:")
print(f"  mapping_id_jadwal          : {len(mapping_id_jadwal)} entri")
print(f"  mapping_id_jadwal_detail   : {len(mapping_id_jadwal_detail)} entri")
print(f"  mapping_id_catatan_kelas   : {len(mapping_id_catatan_kelas)} entri")
print(f"  mapping_id_jadwal_siswa    : {len(mapping_id_jadwal_siswa)} entri")
print(f"  mapping_id_siswa           : {len(mapping_id_siswa)} entri")

# =========================================================
# 2. FUNGSI VALIDASI FK
# =========================================================
def validate_fk(df, col_name, mapping, table_name, fk_name):
    """
    Validasi FK di sebuah DataFrame
    
    Parameters:
    - df: DataFrame yang akan divalidasi
    - col_name: nama kolom FK di df
    - mapping: dictionary mapping old_id -> new_id
    - table_name: nama tabel (untuk output)
    - fk_name: nama FK (untuk output)
    """
    if df is None or len(df) == 0:
        print(f"  ⚠️ {table_name}: DataFrame kosong, skip validasi")
        return
    
    if col_name not in df.columns:
        print(f"  ⚠️ {table_name}: kolom '{col_name}' tidak ditemukan, skip validasi")
        return
    
    # Ambil semua nilai di kolom FK
    fk_values = set(df[col_name].dropna())
    
    if len(fk_values) == 0:
        print(f"  ⚠️ {table_name}: tidak ada data di kolom '{col_name}'")
        return
    
    # Ambil semua key di mapping
    valid_keys = set(mapping.keys())
    
    # Cek berapa yang valid
    invalid = fk_values - valid_keys
    
    print(f"\n  📋 {table_name}.{fk_name}:")
    print(f"     Total baris: {len(df)}")
    print(f"     Total nilai unik FK: {len(fk_values)}")
    
    if len(invalid) == 0:
        print(f"     ✅ SEMUA FK VALID! ({len(fk_values)} nilai)")
        return True
    else:
        print(f"     ❌ {len(invalid)} nilai FK INVALID (tidak ada di mapping)")
        print(f"     Contoh invalid (5): {list(invalid)[:5]}")
        return False

# =========================================================
# 3. VALIDASI SEMUA TABEL
# =========================================================
print("\n" + "="*70)
print("🔍 VALIDASI FK PER TABEL")
print("="*70)

# 3a. Validasi jadwal_detail -> jadwal
if 'jadwal_detail' in data:
    df = data['jadwal_detail']
    validate_fk(df, 'id_jadwal', mapping_id_jadwal, 'jadwal_detail', 'FK ke jadwal')

# 3b. Validasi jadwal_pengajar -> jadwal
if 'jadwal_pengajar' in data:
    df = data['jadwal_pengajar']
    validate_fk(df, 'id_jadwal', mapping_id_jadwal, 'jadwal_pengajar', 'FK ke jadwal')

# 3c. Validasi jadwal_siswa -> jadwal
if 'jadwal_siswa' in data:
    df = data['jadwal_siswa']
    validate_fk(df, 'id_jadwal', mapping_id_jadwal, 'jadwal_siswa', 'FK ke jadwal')

# 3d. Validasi jadwal_siswa -> siswa
if 'jadwal_siswa' in data:
    df = data['jadwal_siswa']
    validate_fk(df, 'id_siswa', mapping_id_siswa, 'jadwal_siswa', 'FK ke siswa')

# 3e. Validasi catatan_kelas -> jadwal
if 'catatan_kelas' in data:
    df = data['catatan_kelas']
    validate_fk(df, 'id_jadwal', mapping_id_jadwal, 'catatan_kelas', 'FK ke jadwal')

# 3f. Validasi catatan_kelas -> jadwal_detail
if 'catatan_kelas' in data:
    df = data['catatan_kelas']
    # Hanya yang id_jadwal_detail tidak null
    df_detail = df[df['id_jadwal_detail'].notna()]
    if len(df_detail) > 0:
        validate_fk(df_detail, 'id_jadwal_detail', mapping_id_jadwal_detail, 'catatan_kelas', 'FK ke jadwal_detail')
    else:
        print(f"\n  📋 catatan_kelas: tidak ada data dengan id_jadwal_detail (semua NULL)")

# 3g. Validasi catatan_kelas_tag -> catatan_kelas
if 'catatan_kelas_tag' in data:
    df = data['catatan_kelas_tag']
    validate_fk(df, 'id_ck', mapping_id_catatan_kelas, 'catatan_kelas_tag', 'FK ke catatan_kelas')

# 3h. Validasi catatan_mingguan -> jadwal (jika ada)
if 'catatan_mingguan' in data:
    df = data['catatan_mingguan']
    if 'id_jadwal' in df.columns:
        validate_fk(df, 'id_jadwal', mapping_id_jadwal, 'catatan_mingguan', 'FK ke jadwal')
    if 'id_jadwal_detail' in df.columns:
        df_detail = df[df['id_jadwal_detail'].notna()]
        if len(df_detail) > 0:
            validate_fk(df_detail, 'id_jadwal_detail', mapping_id_jadwal_detail, 'catatan_mingguan', 'FK ke jadwal_detail')

# 3i. Validasi presensi_siswa (jika ada)
if 'presensi_siswa' in data:
    df = data['presensi_siswa']
    validate_fk(df, 'id_jadwal_detail', mapping_id_jadwal_detail, 'presensi_siswa', 'FK ke jadwal_detail')
    validate_fk(df, 'id_siswa', mapping_id_siswa, 'presensi_siswa', 'FK ke siswa')

# 3j. Validasi catatan_siswa (jika ada)
if 'catatan_siswa' in data:
    df = data['catatan_siswa']
    validate_fk(df, 'id_jadwal', mapping_id_jadwal, 'catatan_siswa', 'FK ke jadwal')
    if 'id_jadwal_detail' in df.columns:
        df_detail = df[df['id_jadwal_detail'].notna()]
        if len(df_detail) > 0:
            validate_fk(df_detail, 'id_jadwal_detail', mapping_id_jadwal_detail, 'catatan_siswa', 'FK ke jadwal_detail')
    validate_fk(df, 'id_siswa', mapping_id_siswa, 'catatan_siswa', 'FK ke siswa')

# 3k. Validasi followup_cs (jika ada)
if 'followup_cs' in data:
    df = data['followup_cs']
    if 'id_cs' in df.columns:
        # Mapping id_cs ke catatan_siswa (jika ada)
        mapping_id_cs = data.get('mapping_id_catatan_siswa', {})
        if mapping_id_cs:
            validate_fk(df, 'id_cs', mapping_id_cs, 'followup_cs', 'FK ke catatan_siswa')
        else:
            print(f"\n  ⚠️ followup_cs: mapping_id_catatan_siswa tidak tersedia")

# =========================================================
# 4. RINGKASAN FINAL
# =========================================================
print("\n" + "="*70)
print("📊 RINGKASAN VALIDASI")
print("="*70)

# Hitung total baris per tabel
print("\n📋 TOTAL BARIS PER TABEL:")
for key, value in data.items():
    if isinstance(value, pd.DataFrame):
        print(f"  {key}: {len(value)} baris")

# Cek tabel yang kosong
empty_tables = [k for k, v in data.items() if isinstance(v, pd.DataFrame) and len(v) == 0]
if empty_tables:
    print(f"\n⚠️ TABEL KOSONG: {empty_tables}")

print("\n✅ Validasi selesai!")

🔍 VALIDASI MAPPING & FOREIGN KEY

📂 Memuat fase_4_afrida.pkl...


✅ Mapping siswa dari mapping_siswa.pkl: 1500 entri

📊 RINGKASAN MAPPING:
  mapping_id_jadwal          : 556 entri
  mapping_id_jadwal_detail   : 17312 entri
  mapping_id_catatan_kelas   : 13733 entri
  mapping_id_jadwal_siswa    : 3958 entri
  mapping_id_siswa           : 1500 entri

🔍 VALIDASI FK PER TABEL

  📋 jadwal_detail.FK ke jadwal:
     Total baris: 17312
     Total nilai unik FK: 556
     ❌ 556 nilai FK INVALID (tidak ada di mapping)
     Contoh invalid (5): [1, 2, 3, 4, 5]

  📋 jadwal_pengajar.FK ke jadwal:
     Total baris: 650
     Total nilai unik FK: 553
     ❌ 553 nilai FK INVALID (tidak ada di mapping)
     Contoh invalid (5): [1, 2, 3, 4, 5]

  📋 jadwal_siswa.FK ke jadwal:
     Total baris: 3958
     Total nilai unik FK: 493
     ❌ 493 nilai FK INVALID (tidak ada di mapping)
     Contoh invalid (5): [1, 2, 3, 4, 5]

  📋 jadwal_siswa.FK ke siswa:
     Total baris: 3958
     Total nilai unik FK: 1318
     ❌ 1318 nilai FK INVALID (tidak ada di mapping)
     Contoh invalid